# 📱 대학생 스마트폰 사용 패턴 × 우울증(PHQ-9) 분석

**데이터**: StudentLife, Dartmouth University 2013  
**분석 내용**:
- 스마트폰 잠금 패턴(횟수, 지속시간)과 PHQ-9 우울증 점수의 상관관계
- 시간대별(심야/오전/오후/저녁/밤) 사용 패턴 분석
- 주중/주말 패턴 비교
- 오후 고/저사용 집단 t-test
- 잠금 지속시간 및 변동성 분석

▶ **셀을 위에서부터 순서대로 실행하세요 (Shift+Enter)**

## STEP 1. 라이브러리 설치 및 한글 폰트

In [ ]:
import subprocess, sys

# 한글 폰트
subprocess.run(['apt-get', '-qq', 'install', 'fonts-nanum'], check=False)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats as sp_stats
import warnings, io, tarfile, urllib.request, os, zipfile, glob
warnings.filterwarnings('ignore')

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

print('✅ 환경 준비 완료')

## STEP 2. 캐글에서 데이터 다운로드 및 추출

**사전 준비**: [kaggle.com](https://www.kaggle.com) → 프로필 → Settings → API → **Create New Token**  
다운로드된 `kaggle.json` 파일을 아래 셀 실행 후 업로드하세요.

In [ ]:
from google.colab import files

# kaggle.json 업로드
print('kaggle.json 파일을 업로드하세요')
files.upload()

# 캐글 인증 설정
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

# 데이터셋 다운로드
print('캐글에서 다운로드 중...')
os.system('pip install -q kaggle')
os.system('kaggle datasets download -d dartweichen/student-life -p /content/')

# 다운로드된 파일 확인
downloaded = glob.glob('/content/*.zip') + glob.glob('/content/*.tar*') + glob.glob('/content/*.bz2')
print(f'다운로드된 파일: {downloaded}')

# 추출 대상 패턴
TARGET_PATTERNS = ['survey/PHQ-9.csv', 'sensing/phonelock/']

def is_target(name):
    for pat in TARGET_PATTERNS:
        if pat in name and name.endswith('.csv'):
            return True
    return False

extracted = {}

for fpath in downloaded:
    if fpath.endswith('.zip'):
        with zipfile.ZipFile(fpath) as z:
            for name in z.namelist():
                if is_target(name):
                    with z.open(name) as f:
                        short = name.split('/')[-1]
                        extracted[short] = pd.read_csv(f)
                        print(f'  {short} ({len(extracted[short])}행)')
    elif '.tar' in fpath or fpath.endswith('.bz2'):
        with tarfile.open(fpath, 'r:*') as tar:
            for member in tar:
                tar.members = []
                if member.isfile() and is_target(member.name):
                    f = tar.extractfile(member)
                    if f:
                        short = member.name.split('/')[-1]
                        extracted[short] = pd.read_csv(io.BytesIO(f.read()))
                        print(f'  {short} ({len(extracted[short])}행)')

print(f'\n✅ 완료 - 총 {len(extracted)}개 파일')

## STEP 3. PHQ-9 전처리

In [ ]:
phq_raw = extracted.get('PHQ-9.csv')
if phq_raw is None:
    raise ValueError('PHQ-9.csv를 찾지 못했습니다. STEP 2를 다시 실행하세요.')

# 텍스트 응답 → 숫자 변환
SCORE_MAP = {
    'Not at all':              0,
    'Several days':            1,
    'More than half the days': 2,
    'Nearly every day':        3,
}

ITEM_COLS = [
    'Little interest or pleasure in doing things',
    'Feeling down, depressed, hopeless.',
    'Trouble falling or staying asleep, or sleeping too much.',
    'Feeling tired or having little energy',
    'Poor appetite or overeating',
    'Feeling bad about yourself or that you are a failure or have let yourself or your family down',
    'Trouble concentrating on things, such as reading the newspaper or watching television',
    'Moving or speaking so slowly that other people could have noticed. Or the opposite being so figety or restless that you have been moving around a lot more than usual',
    'Thoughts that you would be better off dead, or of hurting yourself',
]

# pre(학기 시작 전) 데이터만 사용
phq = phq_raw[phq_raw['type'] == 'pre'].copy()

for col in ITEM_COLS:
    phq[col] = phq[col].map(SCORE_MAP)

phq['phq9_total'] = phq[ITEM_COLS].sum(axis=1)
phq = phq.dropna(subset=['phq9_total'])
phq = phq[(phq['phq9_total'] >= 0) & (phq['phq9_total'] <= 27)]

def severity(s):
    if s <= 4:    return '없음(0-4)'
    elif s <= 9:  return '경미(5-9)'
    elif s <= 14: return '중등도(10-14)'
    elif s <= 19: return '중증(15-19)'
    else:         return '최중증(20-27)'

phq['severity'] = phq['phq9_total'].apply(severity)
phq_final = phq[['uid', 'phq9_total', 'severity']].reset_index(drop=True)

print(f'✅ PHQ-9 전처리 완료: {len(phq_final)}명')
print(phq_final['phq9_total'].describe().round(2))
print('\n심각도 분포:')
print(phq_final['severity'].value_counts())
phq_final.head(10)

## STEP 4. 스마트폰 잠금 패턴 전처리

In [ ]:
lock_list = []
for fname, df in extracted.items():
    if fname == 'PHQ-9.csv':
        continue
    try:
        uid_str = fname.replace('phonelock_', '').replace('.csv', '')  # u00
        df = df.copy()
        df.columns = ['start', 'end']
        df['uid'] = uid_str

        df['start_dt'] = pd.to_datetime(df['start'], unit='s', errors='coerce')
        df['end_dt']   = pd.to_datetime(df['end'],   unit='s', errors='coerce')
        df = df.dropna(subset=['start_dt', 'end_dt'])

        # 잠금 지속시간(초): end - start
        df['lock_duration_sec'] = (df['end'] - df['start']).clip(lower=0)
        df['date'] = df['start_dt'].dt.date

        lock_list.append(df)
    except Exception as e:
        print(f'스킵 {fname}: {e}')

lock_raw = pd.concat(lock_list, ignore_index=True)
print(f'✅ 합산: {len(lock_raw):,}행 / {lock_raw["uid"].nunique()}명')
print(f'기간: {lock_raw["start_dt"].min().date()} ~ {lock_raw["end_dt"].max().date()}')

# 참가자별 일별 집계
daily = lock_raw.groupby(['uid', 'date']).agg(
    lock_count     = ('start',            'count'),
    total_lock_sec = ('lock_duration_sec', 'sum'),
    mean_lock_sec  = ('lock_duration_sec', 'mean')
).reset_index()

# 참가자별 평균
usage = daily.groupby('uid').agg(
    mean_lock_count     = ('lock_count',     'mean'),
    mean_total_lock_min = ('total_lock_sec', 'mean'),
    obs_days            = ('date',           'count')
).reset_index().round(2)

usage['mean_total_lock_min'] = (usage['mean_total_lock_min'] / 60).round(1)

Q1, Q3 = usage['mean_lock_count'].quantile([0.25, 0.75])
IQR = Q3 - Q1
usage['is_outlier'] = (
    (usage['mean_lock_count'] < Q1 - 1.5*IQR) |
    (usage['mean_lock_count'] > Q3 + 1.5*IQR)
)

print(f'\n✅ 사용 패턴 집계: {len(usage)}명  |  이상치: {usage.is_outlier.sum()}명')
usage.head()

## STEP 5. 병합 및 기술통계

In [ ]:
df = pd.merge(phq_final, usage, on='uid', how='inner')
print(f'최종 분석 대상: {len(df)}명\n')

desc = df[['phq9_total', 'mean_lock_count', 'mean_total_lock_min']].describe().round(2)
desc.index = ['N', '평균', '표준편차', '최솟값', 'Q1', '중앙값', 'Q3', '최댓값']
desc.columns = ['PHQ-9 총점', '일평균 잠금 횟수', '일평균 잠금시간(분)']
print(desc.to_string())

print('\n📋 PHQ-9 심각도 분포')
for s, n in df['severity'].value_counts().items():
    pct = n / len(df) * 100
    print(f'  {s}: {n}명 ({pct:.1f}%)')

## STEP 6. 기본 시각화 (4종 + 히트맵)

In [ ]:
SEV_ORDER  = ['없음(0-4)', '경미(5-9)', '중등도(10-14)', '중증(15-19)', '최중증(20-27)']
SEV_COLORS = {'없음(0-4)':'#2ECC71','경미(5-9)':'#F1C40F',
              '중등도(10-14)':'#E67E22','중증(15-19)':'#E74C3C','최중증(20-27)':'#8E44AD'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('StudentLife: 스마트폰 잠금 패턴 × PHQ-9 분석', fontsize=16, fontweight='bold', y=0.98)

# ① PHQ-9 점수 분포
ax = axes[0, 0]
ax.hist(df['phq9_total'], bins=14, color='#4A90D9', edgecolor='white', linewidth=0.8)
ax.axvline(df['phq9_total'].mean(),   color='red',    ls='--', lw=1.8, label=f'평균 {df["phq9_total"].mean():.1f}')
ax.axvline(df['phq9_total'].median(), color='orange', ls='--', lw=1.8, label=f'중앙값 {df["phq9_total"].median():.1f}')
ax.set_xlabel('PHQ-9 총점'); ax.set_ylabel('참가자 수')
ax.set_title('① PHQ-9 점수 분포'); ax.legend()

# ② 심각도별 막대
ax = axes[0, 1]
present = [s for s in SEV_ORDER if s in df['severity'].values]
counts  = [df['severity'].value_counts().get(s, 0) for s in present]
colors  = [SEV_COLORS[s] for s in present]
bars = ax.bar(range(len(present)), counts, color=colors, edgecolor='white')
ax.set_xticks(range(len(present)))
ax.set_xticklabels(present, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('참가자 수'); ax.set_title('② 우울증 심각도 분포')
for b, c in zip(bars, counts):
    if c: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05, str(c), ha='center', va='bottom')

# ③ 산점도 + 회귀선
ax = axes[1, 0]
x, y = df['mean_lock_count'], df['phq9_total']
slope, intercept, r, p, _ = sp_stats.linregress(x, y)
c_list = df['severity'].map(SEV_COLORS)
ax.scatter(x, y, c=c_list, s=55, alpha=0.75, edgecolors='white', lw=0.5)
xr = np.linspace(x.min(), x.max(), 100)
ax.plot(xr, slope*xr+intercept, color='navy', lw=2, label=f'r={r:.3f}  p={p:.3f}')
ax.set_xlabel('일평균 잠금 횟수'); ax.set_ylabel('PHQ-9 총점')
ax.set_title('③ 산점도 + 선형 회귀선'); ax.legend()
sig = '★ 유의미 (p<0.05)' if p < 0.05 else '비유의 (p≥0.05)'
ax.text(0.97, 0.05, sig, transform=ax.transAxes, ha='right', fontsize=9,
        color='navy' if p < 0.05 else 'gray')

# ④ 심각도별 박스플롯
ax = axes[1, 1]
data_bp = [df[df['severity']==s]['mean_lock_count'].values for s in present]
bp = ax.boxplot(data_bp, patch_artist=True, medianprops=dict(color='black', lw=2),
                flierprops=dict(marker='o', markersize=4, color='gray'))
for patch, s in zip(bp['boxes'], present):
    patch.set_facecolor(SEV_COLORS[s]); patch.set_alpha(0.75)
ax.set_xticks(range(1, len(present)+1))
ax.set_xticklabels(present, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('일평균 잠금 횟수'); ax.set_title('④ 심각도별 스마트폰 잠금 빈도')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('분석결과_시각화.png', bbox_inches='tight')
plt.show()
print(f'\n📌 전체 상관계수 r={r:.4f}  |  p={p:.4f}  |  {sig}')

In [ ]:
# 상관관계 히트맵
fig, ax = plt.subplots(figsize=(6, 5))
corr = df[['phq9_total','mean_lock_count','mean_total_lock_min','obs_days']].rename(columns={
    'phq9_total':          'PHQ-9 총점',
    'mean_lock_count':     '평균 잠금 횟수',
    'mean_total_lock_min': '평균 잠금시간(분)',
    'obs_days':            '관측 일수'
}).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, mask=mask, linewidths=0.5, square=True,
            annot_kws={'size':11}, ax=ax)
ax.set_title('상관관계 히트맵', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('상관관계_히트맵.png', bbox_inches='tight')
plt.show()

## STEP 7. 시간대별 심층 분석

In [ ]:
# 시간대 파생변수 추출
lock_raw['hour']       = lock_raw['start_dt'].dt.hour
lock_raw['weekday']    = lock_raw['start_dt'].dt.weekday
lock_raw['is_weekend'] = lock_raw['weekday'] >= 5
lock_raw['time_slot']  = pd.cut(lock_raw['hour'],
    bins=[-1, 5, 11, 17, 20, 23],
    labels=['심야(0-5)', '오전(6-11)', '오후(12-17)', '저녁(18-20)', '밤(21-23)'])

# 참가자 × 시간대별 집계
slot_daily = lock_raw.groupby(['uid', 'date', 'time_slot'], observed=True).size().reset_index(name='count')
slot_usage = slot_daily.groupby(['uid', 'time_slot'], observed=True)['count'].mean().unstack(fill_value=0).reset_index()
slot_usage.columns.name = None

# 주중/주말 분리
weekday_daily = lock_raw[~lock_raw['is_weekend']].groupby(['uid','date']).size().reset_index(name='count')
weekend_daily = lock_raw[ lock_raw['is_weekend']].groupby(['uid','date']).size().reset_index(name='count')
wk = weekday_daily.groupby('uid')['count'].mean().reset_index(name='weekday_mean')
we = weekend_daily.groupby('uid')['count'].mean().reset_index(name='weekend_mean')

# 밤 비율
lock_raw['is_night'] = lock_raw['hour'].isin(range(0, 6)) | lock_raw['hour'].isin(range(21, 24))
night_ratio = lock_raw.groupby('uid').apply(
    lambda x: x['is_night'].sum() / len(x)
).reset_index(name='night_ratio')

# 병합
ext = phq_final.copy()
for d in [slot_usage, wk, we, night_ratio]:
    ext = pd.merge(ext, d, on='uid', how='inner')

print(f'확장 분석 대상: {len(ext)}명')

# 시간대별 시각화
slots = ['심야(0-5)', '오전(6-11)', '오후(12-17)', '저녁(18-20)', '밤(21-23)']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('시간대별 스마트폰 잠금 패턴 × PHQ-9', fontsize=15, fontweight='bold')

for i, slot in enumerate(slots):
    ax = axes[i//3][i%3]
    if slot not in ext.columns or ext[slot].std() == 0:
        ax.set_visible(False); continue
    x, y = ext[slot], ext['phq9_total']
    slope, intercept, r, p, _ = sp_stats.linregress(x, y)
    c_list = ext['severity'].map(SEV_COLORS)
    ax.scatter(x, y, c=c_list, s=50, alpha=0.75, edgecolors='white', lw=0.5)
    xr = np.linspace(x.min(), x.max(), 100)
    ax.plot(xr, slope*xr+intercept, color='navy', lw=1.8)
    sig = '★' if p < 0.05 else ''
    ax.set_title(f'{slot}  r={r:.3f} p={p:.3f} {sig}', fontsize=10)
    ax.set_xlabel('평균 잠금 횟수'); ax.set_ylabel('PHQ-9')

# 주중 vs 주말 박스플롯
ax = axes[1][2]
present = [s for s in SEV_ORDER if s in ext['severity'].values]
x_pos = np.arange(len(present))
w = 0.35
weekday_vals = [ext[ext['severity']==s]['weekday_mean'].values for s in present]
weekend_vals = [ext[ext['severity']==s]['weekend_mean'].values for s in present]
bp1 = ax.boxplot(weekday_vals, positions=x_pos-w/2, widths=0.3, patch_artist=True,
                 medianprops=dict(color='black', lw=2))
bp2 = ax.boxplot(weekend_vals, positions=x_pos+w/2, widths=0.3, patch_artist=True,
                 medianprops=dict(color='black', lw=2))
for patch in bp1['boxes']: patch.set_facecolor('#4A90D9'); patch.set_alpha(0.7)
for patch in bp2['boxes']: patch.set_facecolor('#E74C3C'); patch.set_alpha(0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels(present, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('일평균 잠금 횟수')
ax.set_title('주중(파랑) vs 주말(빨강) × 심각도')
ax.legend([bp1['boxes'][0], bp2['boxes'][0]], ['주중', '주말'], loc='upper right')

plt.tight_layout()
plt.savefig('시간대별_분석.png', bbox_inches='tight')
plt.show()

print('\n=== 시간대별 상관계수 요약 ===')
for slot in slots + ['weekday_mean', 'weekend_mean', 'night_ratio']:
    if slot in ext.columns and ext[slot].std() > 0:
        r, p = sp_stats.pearsonr(ext[slot], ext['phq9_total'])
        sig = '★ 유의' if p < 0.05 else ''
        print(f'  {slot:20s}  r={r:+.3f}  p={p:.3f}  {sig}')

## STEP 8. 오후 시간대 심층 분석 (t-test)

In [ ]:
# 오후 사용량 기준 중앙값으로 고/저 집단 분리
median_afternoon = ext['오후(12-17)'].median()
ext['afternoon_group'] = np.where(ext['오후(12-17)'] > median_afternoon, '고사용', '저사용')

high = ext[ext['afternoon_group'] == '고사용']['phq9_total']
low  = ext[ext['afternoon_group'] == '저사용']['phq9_total']

t_stat, p_val = sp_stats.ttest_ind(high, low)
pooled_std = np.sqrt((high.std()**2 + low.std()**2) / 2)
cohens_d = (high.mean() - low.mean()) / pooled_std

print('=== 오후 사용량 집단별 PHQ-9 비교 ===')
print(f'고사용 집단 (n={len(high)}): 평균 {high.mean():.2f}, 표준편차 {high.std():.2f}')
print(f'저사용 집단 (n={len(low)}):  평균 {low.mean():.2f}, 표준편차 {low.std():.2f}')
print(f't = {t_stat:.3f},  p = {p_val:.3f}  {"★ 유의 (p<0.05)" if p_val < 0.05 else "비유의"}')
print(f"Cohen's d = {cohens_d:.3f}  ({'대' if abs(cohens_d)>0.8 else '중' if abs(cohens_d)>0.5 else '소'} 효과크기)")

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('오후(12-17시) 스마트폰 사용량 × PHQ-9 심층 분석', fontsize=13, fontweight='bold')

# 박스플롯
ax = axes[0]
bp = ax.boxplot([low.values, high.values], patch_artist=True,
                medianprops=dict(color='black', lw=2),
                flierprops=dict(marker='o', markersize=4))
bp['boxes'][0].set_facecolor('#2ECC71'); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#E74C3C'); bp['boxes'][1].set_alpha(0.7)
ax.set_xticks([1, 2])
ax.set_xticklabels(['저사용\n(오후 폰 적게)', '고사용\n(오후 폰 많이)'])
ax.set_ylabel('PHQ-9 총점')
ax.set_title(f't-test: t={t_stat:.3f}, p={p_val:.3f}')
sig_txt = '★ p<0.05' if p_val < 0.05 else 'n.s.'
ax.text(1.5, ax.get_ylim()[1]*0.95, sig_txt, ha='center', fontsize=12,
        color='red' if p_val < 0.05 else 'gray')

# 바이올린 플롯
ax = axes[1]
parts = ax.violinplot([low.values, high.values], positions=[1, 2], showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(['#2ECC71', '#E74C3C'][i]); pc.set_alpha(0.6)
ax.set_xticks([1, 2])
ax.set_xticklabels(['저사용', '고사용'])
ax.set_ylabel('PHQ-9 총점')
ax.set_title('분포 형태 비교 (바이올린)')

# 심각도 구성 비교
ax = axes[2]
low_pct  = ext[ext['afternoon_group']=='저사용']['severity'].value_counts(normalize=True).reindex(SEV_ORDER, fill_value=0) * 100
high_pct = ext[ext['afternoon_group']=='고사용']['severity'].value_counts(normalize=True).reindex(SEV_ORDER, fill_value=0) * 100
x = np.arange(len(SEV_ORDER))
ax.bar(x - 0.2, low_pct,  width=0.4, label='저사용', color='#2ECC71', alpha=0.8)
ax.bar(x + 0.2, high_pct, width=0.4, label='고사용', color='#E74C3C', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(SEV_ORDER, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('비율 (%)')
ax.set_title('심각도 구성 비교')
ax.legend()

plt.tight_layout()
plt.savefig('오후사용_심층분석.png', bbox_inches='tight')
plt.show()

## STEP 9. 잠금 지속시간 분석

In [ ]:
duration_daily = lock_raw.groupby(['uid', 'date']).agg(
    mean_dur_sec  = ('lock_duration_sec', 'mean'),
    total_dur_sec = ('lock_duration_sec', 'sum'),
).reset_index()

duration_user = duration_daily.groupby('uid').agg(
    mean_dur_min  = ('mean_dur_sec',  lambda x: (x/60).mean()),
    total_dur_min = ('total_dur_sec', lambda x: (x/60).mean()),
    cv_dur        = ('mean_dur_sec',  lambda x: x.std()/x.mean())
).reset_index().round(3)

ext2 = pd.merge(ext, duration_user, on='uid', how='inner')
print(f'지속시간 분석 대상: {len(ext2)}명')

print('\n=== 잠금 지속시간 × PHQ-9 상관계수 ===')
for col, label in [('mean_dur_min',  '평균 잠금시간(분)'),
                   ('total_dur_min', '일평균 총 잠금시간(분)'),
                   ('cv_dur',        '잠금시간 변동성(CV)')]:
    if ext2[col].std() > 0:
        r, p = sp_stats.pearsonr(ext2[col], ext2['phq9_total'])
        sig = '★ 유의' if p < 0.05 else ''
        print(f'  {label:25s}  r={r:+.3f}  p={p:.3f}  {sig}')

# 시각화
SEV_COLORS_MAP = {'없음(0-4)':'#2ECC71','경미(5-9)':'#F1C40F',
                  '중등도(10-14)':'#E67E22','중증(15-19)':'#E74C3C','최중증(20-27)':'#8E44AD'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('잠금 지속시간 × PHQ-9 분석', fontsize=13, fontweight='bold')

for i, (col, label) in enumerate([
    ('mean_dur_min',  '평균 잠금 지속시간(분)'),
    ('total_dur_min', '일평균 총 잠금시간(분)'),
    ('cv_dur',        '잠금시간 변동성(CV)'),
]):
    ax = axes[i]
    x, y = ext2[col], ext2['phq9_total']
    slope, intercept, r, p, _ = sp_stats.linregress(x, y)
    c_list = ext2['severity'].map(SEV_COLORS_MAP)
    ax.scatter(x, y, c=c_list, s=55, alpha=0.75, edgecolors='white', lw=0.5)
    xr = np.linspace(x.min(), x.max(), 100)
    ax.plot(xr, slope*xr+intercept, color='navy', lw=2)
    sig = '★' if p < 0.05 else ''
    ax.set_title(f'{label}\nr={r:+.3f}  p={p:.3f}  {sig}', fontsize=10)
    ax.set_xlabel(label); ax.set_ylabel('PHQ-9 총점')

plt.tight_layout()
plt.savefig('잠금지속시간_분석.png', bbox_inches='tight')
plt.show()

## STEP 10. 결과 저장 및 깃허브 업로드

In [ ]:
# 분석 데이터 저장
df.to_csv('분석데이터_최종.csv', index=False, encoding='utf-8-sig')
desc.to_csv('기술통계.csv', encoding='utf-8-sig')

output_files = [
    '분석데이터_최종.csv', '기술통계.csv',
    '분석결과_시각화.png', '상관관계_히트맵.png',
    '시간대별_분석.png', '오후사용_심층분석.png', '잠금지속시간_분석.png'
]

print('저장 완료:')
for f in output_files:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f'  ✔ {f}  ({size:,} bytes)')

In [ ]:
# STEP 10. 결과 파일 다운로드
from google.colab import files

output_files = [
    '분석데이터_최종.csv',
    '기술통계.csv',
    '분석결과_시각화.png',
    '상관관계_히트맵.png',
    '시간대별_분석.png',
    '오후사용_심층분석.png',
    '잠금지속시간_분석.png',
]

print('저장된 파일 목록:')
for f in output_files:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f'  ✔ {f}  ({size:,} bytes)')

print('\n📥 파일 다운로드 시작...')
for f in output_files:
    if os.path.exists(f):
        files.download(f)

print('\n✅ 완료!')

## 결과 요약

| 분석 | 결과 |
|---|---|
| 전체 잠금 횟수 × PHQ-9 | r=-0.092, p=0.541 (비유의) |
| 오후(12-17시) × PHQ-9 | r=-0.355, **p=0.015 ★** |
| 오후 고/저사용 t-test | t=-2.272, **p=0.028 ★**, Cohen's d=0.670 (중간 효과) |
| 잠금 지속시간 × PHQ-9 | 모두 비유의 |

**핵심 해석**: 단순 사용량보다 **오후 시간대**의 스마트폰 사용 패턴이 우울증과 유의미한 관련을 보임.  
오후 고사용 집단(PHQ-9 평균 4.04)이 저사용 집단(7.00)보다 유의미하게 낮은 우울 점수를 나타냄.